# AIHA scRNA-seq Drug Repurposing Pipeline
**Dataset:** GSE301528 — Bone marrow aspirate, Autoimmune Hemolytic Anemia  
**Conditions:** Diagnosis (n=5) · Remission (n=3) · Relapse/Refractory (n=4)  
**Tissue:** Bone marrow (whole BM; sorted subpopulations excluded)  
**Repo:** github.com/glenritschel/aiha-scrna  
**Output:** /content/drive/MyDrive/Ritschel_Research/aiha_scrna_output

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
%%capture
!pip install scanpy anndata scvi-tools gseapy igraph leidenalg

In [ ]:
import os, subprocess

REPO_DIR = '/content/aiha-scrna'
if not os.path.exists(f'{REPO_DIR}/src/01_load_qc.py'):
    subprocess.run(['git', 'clone',
                    'https://github.com/glenritschel/aiha-scrna.git',
                    REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print("Repo up to date.")

import sys
sys.path.insert(0, f'{REPO_DIR}/src')
print(f"Scripts: {sorted(os.listdir(REPO_DIR + '/src'))}")

In [ ]:
import os, gc, re, time, json, random
import numpy as np
import pandas as pd
import scipy.io
import anndata as ad
import scanpy as sc

sc.settings.verbosity = 1

DRIVE_BASE = "/content/drive/MyDrive/Ritschel_Research/aiha_scrna_output"
RAW_DIR    = os.path.join(DRIVE_BASE, "raw")
PROCESSED  = os.path.join(DRIVE_BASE, "processed")
os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

print(f"RAW_DIR   : {RAW_DIR}")
print(f"PROCESSED : {PROCESSED}")
if os.path.exists(RAW_DIR):
    files = os.listdir(RAW_DIR)
    print(f"Files in raw: {len(files)}")

In [ ]:
# Download GSE301528 raw data (only if not already present)
import os

tar_path = os.path.join(RAW_DIR, "GSE301528_RAW.tar")
if not os.path.exists(tar_path) or os.path.getsize(tar_path) < 1e8:
    print("Downloading GSE301528_RAW.tar (~663 MB)...")
    !wget -q -P {RAW_DIR} \
        "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE301nnn/GSE301528/suppl/GSE301528_RAW.tar"
    print("Extracting...")
    !tar -xf {tar_path} -C {RAW_DIR}
    print("Done.")
else:
    print("Data already downloaded.")

# Verify whole-BM files present (sorted subpops excluded by condition map)
mtx_files = [f for f in os.listdir(RAW_DIR) if f.endswith('_matrix.mtx.gz')]
print(f"MTX files in raw: {len(mtx_files)}")

## Script 01 — Load & QC
*Loads 12 whole-BM samples (sorted subpops excluded), QC filter, normalise, HVG.*

In [ ]:
# Skip guard: reuse if already saved
import os
OUT_01 = os.path.join(PROCESSED, "01_loaded.h5ad")
if os.path.exists(OUT_01):
    print("01_loaded.h5ad found — skipping Script 01.")
else:
    %run {REPO_DIR}/src/01_load_qc.py

## Script 02 — scVI Embedding & Clustering
*Train scVI (200 epochs, T4 GPU), checkpoint after training, UMAP + multi-resolution Leiden.*

In [ ]:
OUT_02 = os.path.join(PROCESSED, "02_scvi.h5ad")
if os.path.exists(OUT_02):
    print("02_scvi.h5ad found — skipping Script 02.")
else:
    %run {REPO_DIR}/src/02_scvi_embed.py

In [ ]:
# Resume from checkpoint if session reset after training
CKPT_02 = os.path.join(PROCESSED, "02_scvi_ckpt.h5ad")
OUT_02   = os.path.join(PROCESSED, "02_scvi.h5ad")

if not os.path.exists(OUT_02) and os.path.exists(CKPT_02):
    print("Loading scVI checkpoint — completing UMAP + Leiden...")
    import torch, random, scvi
    RANDOM_SEED = 0
    np.random.seed(RANDOM_SEED); random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
    adata = sc.read_h5ad(CKPT_02)
    print(f"  {adata.n_obs:,} cells, X_scVI: {'X_scVI' in adata.obsm}")
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata)
    for res in [0.5, 0.8, 1.2]:
        sc.tl.leiden(adata, resolution=res, key_added=f"leiden_{res}",
                     random_state=0, flavor="igraph", n_iterations=2, directed=False)
        print(f"  Resolution {res}: {adata.obs[f'leiden_{res}'].nunique()} clusters")
    adata.obs["leiden"] = adata.obs["leiden_0.5"].copy()
    adata.write_h5ad(OUT_02)
    print(f"Saved: {OUT_02}")

## Script 03 — Cell Type Annotation
*12 bone marrow cell types: HSC/progenitors, erythroid lineage, myeloid, lymphoid, plasma.*

In [ ]:
OUT_03 = os.path.join(PROCESSED, "03_annotated.h5ad")
if os.path.exists(OUT_03):
    print("03_annotated.h5ad found — skipping Script 03.")
else:
    %run {REPO_DIR}/src/03_annotate_clusters.py

## Script 04 — AIHA Signature Scoring
*5 AIHA signatures: erythrophagocytosis stress, complement/FcR activation, T cell exhaustion, inflammatory cytokines, type I IFN.*

In [ ]:
OUT_04 = os.path.join(PROCESSED, "04_scored.h5ad")
if os.path.exists(OUT_04):
    print("04_scored.h5ad found — skipping Script 04.")
else:
    %run {REPO_DIR}/src/04_signature_scoring.py

## Script 05 — Differential Expression
*Primary: Diagnosis vs Remission · Secondary: RR vs Remission · Tertiary: Diagnosis vs RR · Erythroid + T cell subsets.*

In [ ]:
OUT_05 = os.path.join(PROCESSED, "05_de.h5ad")
if os.path.exists(OUT_05):
    print("05_de.h5ad found — skipping Script 05.")
else:
    %run {REPO_DIR}/src/05_differential_expression.py

## Script 06 — LINCS L1000 Reversal Scoring
*Enrichr queries across all DE comparisons and cluster-level profiles. Expect 20–40 minutes.*

In [ ]:
OUT_06 = os.path.join(PROCESSED, "06_lincs.csv")
if os.path.exists(OUT_06):
    print("06_lincs.csv found — skipping Script 06.")
else:
    %run {REPO_DIR}/src/06_lincs_repurposing.py

## Script 07 — Novelty Prioritization
*PubMed AIHA/hematologic autoimmune/complement novelty filtering. Priority score = reversal × queries × multiplier.*

In [ ]:
OUT_07 = os.path.join(PROCESSED, "priority_candidates.csv")
if os.path.exists(OUT_07):
    print("priority_candidates.csv found — skipping Script 07.")
else:
    %run {REPO_DIR}/src/07_novelty_prioritization.py

## Results Summary

In [ ]:
import pandas as pd, os

pc_path = os.path.join(PROCESSED, "priority_candidates.csv")
pw_path = os.path.join(PROCESSED, "patent_watch.csv")

if os.path.exists(pc_path):
    pc = pd.read_csv(pc_path)
    print(f"Total candidates: {len(pc)}")
    print(f"\nNovelty breakdown:")
    print(pc['novelty_tier'].value_counts().to_string())
    print(f"\nTop 10 priority candidates:")
    print(pc.head(10)[['compound','moa','novelty_tier',
                        'max_reversal_score','n_queries','priority_score']].to_string(index=False))

if os.path.exists(pw_path):
    pw = pd.read_csv(pw_path)
    print(f"\nPatent watch (NOVEL_ALL): {len(pw)} compounds")
    print(pw[['compound','moa','priority_score']].to_string(index=False))